# Task 2 and Task 3 Analysis Evidence - xfan0282


## Member Scope

This notebook shows the Task 2 and Task 3 workflow for `xfan0282`. The selected SA4 is read from `configs/local.yaml`, then the settings are restricted to this member's SA4 only.


In [ ]:
from dataclasses import replace

import pandas as pd

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table
from data2001.task4.maps import build_score_choropleth_map, build_poi_density_choropleth_map, build_poi_point_scatter_map

from data2001.task4.queries import (
    load_api_extraction_summary,
    load_correlation_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
)


MEMBER_UNIKEY = "xfan0282"

base_settings = load_settings("configs/local.yaml")
member_sa4 = (base_settings.task2_import.selected_sa4_by_member.get(MEMBER_UNIKEY) or "").strip()
if not member_sa4:
    raise ValueError(f"No SA4 configured for {MEMBER_UNIKEY}. Fill configs/local.yaml selected_sa4_by_member.")

settings = replace(
    base_settings,
    task2_import=replace(
        base_settings.task2_import,
        crawl_scope="selected_sa4",
        selected_sa4_by_member={MEMBER_UNIKEY: member_sa4},
    ),
    task3_score=replace(base_settings.task3_score, score_universe="selected_sa4"),
)
engine = create_engine_from_settings(settings.database)

member_scope = pd.DataFrame([{"unikey": MEMBER_UNIKEY, "selected_sa4": member_sa4}])
display(member_scope)

## Single-SA4 Full Workflow Run

This cell runs the full personal SA4 workflow. It clears the local database, then downloads and rebuilds the data for the SA4 selected by `xfan0282`. The full workflow usually takes about 5-6 seconds on my machine.


In [ ]:
workflow_steps = [
    "init_db",
    "clear_db",
    "import_boundaries",
    "import_poi",
    "import_income",
    "compute_score",
]

workflow_summary = execute_workflow_steps(
    engine,
    settings,
    workflow_steps,
    title=f"{MEMBER_UNIKEY} single-SA4 full rebuild",
)
display(workflow_summary)

## Single-SA4 Database Verification

This section checks whether the database now only contains the current member's selected SA4. After the full rebuild, the SA2 table should only list `Sydney - City and Inner South`.


In [ ]:
schema = settings.database.schema_name

display(pd.read_sql(
    f"""
    SELECT sa4_name, COUNT(*) AS sa2_count
    FROM {schema}.sa2
    GROUP BY sa4_name
    """,
    engine,
))

## Task 2 Evidence: API Extraction and Spatial Join

Task 2 extracts POI data for each SA2 in the selected SA4. The workflow first uses the ABS boundary API to get the SA2 polygons and bounding boxes inside the current SA4. It then calls the NSW POI API once for each SA2 bounding box.

The bounding box is only used to collect candidate POIs. It is a rectangle, so it can include points outside the real SA2 boundary. For this reason, the final SA2 assignment is not based directly on the bounding box. Cleaned POIs are stored in `poi_clean`, then PostGIS spatial join assigns them to SA2 polygons and writes the result into `sa2_poi`.


In [ ]:
display(load_api_extraction_summary(settings))
display(load_spatial_join_summary(engine, settings))

## Task 3 Evidence: Score Calculation

Task 3 calculates the well-resourced score from the number of POIs assigned to each SA2. The workflow calculates the mean and standard deviation of POI counts in the current score universe. Then it converts each SA2's POI count into a z-score, applies the sigmoid function, and multiplies the result by 100 to get `score_100`.

In this notebook, the score universe is the selected SA4 for `xfan0282`. So the score is a relative score inside this SA4, not an absolute score for all of Greater Sydney. The configuration also excludes SA2s with population below 100. This is why the database has 27 SA2 rows, but the final score table has 25 scored SA2s.


In [ ]:
display(load_score_input_summary(engine, settings))

scores = load_sa2_scores(engine, settings)
display(scores.head())
display(build_top_bottom_table(scores, n=settings.charts.top_n))

## Individual Visual Analysis

This section explains the score distribution, top and bottom SA2s, spatial patterns, POI group structure, and the relationship between score and median income for this member's SA4.


In [ ]:
poi_groups = load_poi_group_counts(engine, settings)
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit)
score_income = load_score_income(engine, settings)

### Score Distribution

This histogram shows the distribution of well-resourced scores across SA2s in Sydney - City and Inner South. The x-axis is `score_100`, from 0 to 100. The y-axis is the number of SA2s in each score range.

Most SA2s sit in the low-to-middle or middle score range, while a small number of SA2s are much higher than the rest. This suggests that POIs are not evenly distributed inside this SA4. A few areas have unusually high POI counts, which pushes their scores up.


In [ ]:
build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show()

### Top and Bottom SA2 Scores

These two bar charts show the highest and lowest scoring SA2s. The x-axis is `score_100`, and the y-axis is the SA2 name. The colour shows the SA4. Since this notebook only analyses one SA4, the colour is mainly kept for a consistent chart format.

The two highest scoring SA2s are Sydenham - Tempe - St Peters and Sydney (North) - Millers Point, with scores of 97.26 and 95.01. They are much higher than the other SA2s. After checking the POI point map, these high scores seem to come from very dense records of specific POI types, not from a balanced mix of all resource types.


In [ ]:
build_top_sa2_bar(scores, n=settings.charts.top_n).show()
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show()

### SQL Evidence: Top and Bottom Score Values

The SQL below checks the high-score SA2s, `score_100`, POI count, and population used in the explanation above.


In [ ]:
score_rank_sql = f"""
SELECT
    s.sa2_name,
    sc.poi_count,
    ROUND(sc.score_100::numeric, 2) AS score_100,
    s.population
FROM {schema}.sa2_score sc
JOIN {schema}.sa2 s
  ON s.sa2_code = sc.sa2_code
WHERE s.sa4_name = %(member_sa4)s
ORDER BY sc.score_100 DESC
LIMIT 5
"""

display(pd.read_sql(score_rank_sql, engine, params={"member_sa4": member_sa4}))


### Score Choropleth Map

This choropleth map shows the spatial distribution of well-resourced scores by SA2. Darker colours mean higher scores. The hover data includes SA2 name, population, POI count, `z_poi`, and score.

The map helps show whether high scores are spatially clustered. Sydenham - Tempe - St Peters and Millers Point appear as clear high-score areas. The POI point map later shows that these high scores are mainly caused by dense records of specific POI types.


In [ ]:
build_score_choropleth_map(scores).show()

### Population-Adjusted POI Density Map

Each SA2 polygon is coloured by `poi_per_1000`, which means the number of assigned POIs per 1,000 residents. The value is calculated from `poi_count / population * 1000`.

In [ ]:
build_poi_density_choropleth_map(scores).show()


### POI Point Map

This scatter map shows the locations of cleaned POIs. Each point is one POI, and the colour shows the POI group, such as Transport, Recreation, or Community.

A clear Transport line can be seen from Sydenham - Tempe - St Peters towards Newtown and Camperdown - Darlington. The SQL check shows that Sydenham - Tempe - St Peters has 310 Roadside Emergency Telephone POIs, Newtown has 37, and Camperdown - Darlington has 12. This line pattern is mainly caused by roadside emergency telephones, not by a wide mix of public resources.

There is also a regular group of wharf POIs near Millers Point. The SQL check shows that Sydney (North) - Millers Point has 74 Wharf POIs, which is an important reason for its high POI count.

In several south-eastern SA2s, the share of Recreation POIs is clearly higher. For example, Recreation POIs make up 69.1% of Pagewood - Hillsdale - Daceyville, 61.4% of Mascot, and 56.5% of Rosebery - Beaconsfield. Many of these POIs are parks or sports-related locations, so different areas have different POI structures.


In [ ]:
build_poi_point_scatter_map(poi_points).show()

### SQL Evidence: POI Point-Map Patterns

The SQL below checks the Roadside Emergency Telephone pattern, the Wharf pattern, and the Recreation POI share in the south-eastern SA2s.


In [ ]:
roadside_sql = f"""
SELECT
    s.sa2_name,
    p.poigroup_name,
    p.poitype,
    COUNT(*) AS poi_count
FROM {schema}.poi_clean p
JOIN {schema}.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN {schema}.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE p.poitype = 'Roadside Emergency Telephone'
GROUP BY s.sa2_name, p.poigroup_name, p.poitype
ORDER BY poi_count DESC, s.sa2_name
LIMIT 10
"""

wharf_sql = f"""
SELECT
    s.sa2_name,
    p.poigroup_name,
    p.poitype,
    COUNT(*) AS poi_count
FROM {schema}.poi_clean p
JOIN {schema}.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN {schema}.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE p.poitype = 'Wharf'
GROUP BY s.sa2_name, p.poigroup_name, p.poitype
ORDER BY poi_count DESC, s.sa2_name
LIMIT 10
"""

recreation_share_sql = f"""
WITH counts AS (
    SELECT
        s.sa2_name,
        COUNT(*) AS total_poi,
        COUNT(*) FILTER (WHERE p.poigroup_name = 'Recreation') AS recreation_poi
    FROM {schema}.poi_clean p
    JOIN {schema}.sa2_poi sp
      ON sp.poi_objectid = p.objectid
    JOIN {schema}.sa2 s
      ON s.sa2_code = sp.sa2_code
    WHERE s.sa2_name IN (
        'Pagewood - Hillsdale - Daceyville',
        'Mascot',
        'Rosebery - Beaconsfield',
        'Zetland',
        'Eastlakes',
        'Botany',
        'Waterloo',
        'Banksmeadow'
    )
    GROUP BY s.sa2_name
)
SELECT
    sa2_name,
    total_poi,
    recreation_poi,
    ROUND(recreation_poi::numeric / NULLIF(total_poi, 0) * 100, 1) AS recreation_pct
FROM counts
ORDER BY recreation_pct DESC
"""

display(pd.read_sql(roadside_sql, engine))
display(pd.read_sql(wharf_sql, engine))
display(pd.read_sql(recreation_share_sql, engine))


### POI Group Distribution

This chart shows the number of POIs in each POI group for the current SA4. The x-axis is POI count, and the y-axis is POI group.

Transport, Recreation, and Community are the largest POI groups, with 555, 481, and 465 POIs. The high Transport count partly reflects dense Roadside Emergency Telephone and Wharf records, so the score may be sensitive to repeated infrastructure-style POIs.


In [ ]:
build_poi_group_distribution(poi_groups).show()

### SQL Evidence: POI Group Totals

The SQL below checks the Transport, Recreation, and Community POI counts used in the explanation above.


In [ ]:
poi_group_sql = f"""
SELECT
    p.poigroup_name,
    COUNT(*) AS poi_count
FROM {schema}.poi_clean p
JOIN {schema}.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN {schema}.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE s.sa4_name = %(member_sa4)s
GROUP BY p.poigroup_name
ORDER BY poi_count DESC
"""

print(poi_group_sql)
display(pd.read_sql(poi_group_sql, engine, params={"member_sa4": member_sa4}))


### Score and Median Income

This scatter plot shows the relationship between median income and well-resourced score. The x-axis is `median_income_2022_23`, the y-axis is `score_100`, the point size shows POI count, and the colour shows SA4.

The plot does not show a clear increase in score as income increases. Pearson correlation is 0.0811 with a p-value of 0.7062. Spearman correlation is 0.3112 with a p-value of 0.1388. Both are not statistically significant, so the relationship between median income and score is weak in this SA4 sample.

Sydenham - Tempe - St Peters and Millers Point have much higher scores than other SA2s at ordinary or similar income levels. Based on the POI type checks above, this is more likely caused by dense Roadside Emergency Telephone and Wharf records than by income itself.


In [ ]:
build_score_income_scatter(score_income).show()

### SQL Evidence: Income-Score Correlation

The SQL below checks the Pearson/Spearman correlation results and the high-score SA2s in the income-score scatter plot.


In [ ]:
correlation_sql = f"""
SELECT
    method,
    ROUND(statistic::numeric, 4) AS statistic,
    ROUND(p_value::numeric, 4) AS p_value,
    n,
    alpha,
    is_significant
FROM {schema}.score_income_correlation
ORDER BY method
"""

top_income_sql = f"""
SELECT
    s.sa2_name,
    sc.poi_count,
    ROUND(sc.score_100::numeric, 2) AS score_100,
    i.median_income_2022_23,
    i.income_earners_2022_23
FROM {schema}.sa2_score sc
JOIN {schema}.sa2 s
  ON s.sa2_code = sc.sa2_code
JOIN {schema}.sa2_income i
  ON i.sa2_code = sc.sa2_code
WHERE s.sa4_name = %(member_sa4)s
ORDER BY sc.score_100 DESC
LIMIT 5
"""

display(pd.read_sql(correlation_sql, engine))
display(pd.read_sql(top_income_sql, engine, params={"member_sa4": member_sa4}))


## Correlation and Interpretation Notes

This section summarises the correlation tests between score and median income. Pearson measures the linear relationship, while Spearman measures the rank-based relationship. In the final interpretation, both the statistic and p-value should be considered. If the p-value is above 0.05, the result should be described as not statistically significant, rather than saying that income has no effect.


In [ ]:
display(load_correlation_summary(engine, settings))

## Final Key Findings

- The selected SA4 for `xfan0282` is `Sydney - City and Inner South`. This personal workflow rebuilds the database for this SA4 only, so the results should be interpreted as member-level evidence rather than a full Greater Sydney result.
- The workflow contains 27 SA2 areas in the selected SA4. After applying the population filter, 25 SA2 areas are included in the final score table.
- The two highest scoring SA2s are `Sydenham - Tempe - St Peters` and `Sydney (North) - Millers Point`, with scores of 97.26 and 95.01. These scores are much higher than the rest of the SA4.
- The high scores appear to be driven by dense clusters of specific POI types rather than a balanced mix of resources. `Sydenham - Tempe - St Peters` is strongly affected by Roadside Emergency Telephone POIs, while `Sydney (North) - Millers Point` is strongly affected by Wharf POIs.
- The score-income relationship is not statistically significant in this SA4 sample. Pearson correlation is 0.0811 with p-value 0.7062, and Spearman correlation is 0.3112 with p-value 0.1388. This suggests that median income alone does not explain the score pattern here.
